# Molecular dynamics of an explicitly solvated small protein

For the simulations in this notebook, we will make use of the crystal structure of the Villin Headpiece subdomain [1YRF](https://www.rcsb.org/structure/1YRF). This is a relatively small protein (35 residues) consisting of three alpha helices. This is a prototypical fast folding protein, see [10.1016/j.jmb.2006.03.034](https://doi.org/10.1016/j.jmb.2006.03.034) and [10.1073/pnas.0502495102](https://doi.org/10.1073/pnas.0502495102), and it is therefore a popular benchmark for protein folding molecular dynamics simulations, e.g. [10.1073/pnas.1800690115](https://doi.org/10.1073/pnas.1800690115) (force field accuracy) and [10.1109/SC.2014.9](https://doi.org/10.1109/SC.2014.9) (computational performance).

If you want to run a longer simulation in this notebook, after increasing the number of time steps below, it may be useful to submit this notebook as a non-interactive job.
This is explained in section `03_noninteractive` of the tutorial.

In [ ]:
import shutil
from sys import stdout

import matplotlib.pyplot as plt
import mdtraj
import nglview
import pandas
from openmm import *
from openmm.app import *
from openmm.unit import *

The PDB file for 1YRF contains more information than we need. It holds several possible states of some residues mixed into one file and OpenMM cannot handle this. Most atomic positions are the same for all these states, but a few have an extra `A`, `B`, `C`, ... just after the atom name. The following function splits such a PDB file into multiple ones. Each of these files is suitable for starting an OpenMM simulation.

In [ ]:
def split_pdb(fn_pdb):
    pos = 16
    groups = {}
    counter = 0
    with open(fn_pdb) as f:
        for line in f:
            if line.startswith("ATOM"):
                state = line[pos]
                line = line[:pos] + " " + line[pos + 1 :]
                groups.setdefault(state, []).append((counter, line))
                counter += 1
    print("Group keys:", groups.keys())
    for key, lines_group in groups.items():
        if key == " ":
            continue
        lines_both = lines_group + groups[" "]
        lines_both.sort()
        new_fn_pdb = f"{fn_pdb[:-4]}_{key.lower()}.pdb"
        with open(new_fn_pdb, "w") as f:
            for _counter, line in lines_both:
                f.write(line)
        print("Written", new_fn_pdb)


split_pdb("1yrf.pdb")

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**
> Inspect the contents of the files `1yrf_a.pdb` and `1yrf_b.pdb`. Verify for at least one atom that these files contain the correct information taken from `1yrf.pdb`.

The energy minimization in the next cell is monitored with a minimization reporter, as explained in the first notebook of this tutorial (`01_first_steps/01_water.ipynb`).
The reporter below is a bit more elaborate, because the minimization of a solvated protein is more complicated than that of a single water molecule:

- The system contains constraints (`constraints=HBonds` below), but the L-BFGS algorithm cannot handle constraints.
  The minimizer therefore replaces them by harmonic restraints and minimizes the sum of the potential energy and the restraint energy.
  If the constraints are not satisfied well enough afterwards, the restraints are made stronger and the minimization is repeated.
  The reporter stores the `restraint strength`, to see when this happens.

- Every such repetition starts counting iterations from zero again, and `maxIterations` applies to each repetition separately.
  The reporter therefore keeps its own counter to obtain a continuous iteration number.

In [ ]:
class CSVMinimizationReporter(MinimizationReporter):
    """Write the energy after every minimization iteration to a CSV file."""

    def __init__(self, filename):
        super().__init__()
        self.file = open(filename, "w")
        self.file.write("Iteration,Energy (kJ/mole),Restraint strength\n")
        self.counter = 0

    def report(self, iteration, x, grad, args):
        strength = args["restraint strength"]
        self.file.write(f"{self.counter},{args['system energy']},{strength}\n")
        self.counter += 1
        # Returning True would stop the minimization.
        return False

    def close(self):
        self.file.close()

In [ ]:
pdb = PDBFile("1yrf_a.pdb")
modeller = Modeller(pdb.topology, pdb.positions)
forcefield = ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
modeller.addHydrogens(forcefield)
modeller.addSolvent(forcefield, model="tip3p", padding=1 * nanometer)
system = forcefield.createSystem(modeller.topology, nonbondedMethod=PME, constraints=HBonds)
temperature = 300 * kelvin
pressure = 1 * bar
integrator = LangevinIntegrator(temperature, 1 / picosecond, 2 * femtoseconds)
system.addForce(MonteCarloBarostat(pressure, temperature))
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)
reporter = CSVMinimizationReporter("minimization.csv")
simulation.minimizeEnergy(maxIterations=100, reporter=reporter)
reporter.close()
positions = simulation.context.getState(getPositions=True).getPositions()
with open("init.pdb", "w") as f:
    PDBFile.writeFile(simulation.topology, positions, f)

The progress of the minimization can be plotted in the same way as in the first notebook.
The dashed vertical lines mark the iterations where the restraint strength was increased.

In [ ]:
df_min = pandas.read_csv("minimization.csv")
plt.close("minimization")
fig, ax = plt.subplots(num="minimization")
df_min.plot(kind="line", x="Iteration", y="Energy (kJ/mole)", ax=ax)
# Mark the iterations where the restraint strength increases.
for iteration in df_min["Iteration"][df_min["Restraint strength"].diff() > 0]:
    ax.axvline(iteration, color="gray", linestyle="--")

Within each repetition, the energy decreases rapidly at first, and then more slowly.
Note that this is the potential energy of the system, without the restraint energy.
It jumps up when the restraints become stronger, because the geometry is then adjusted to satisfy the constraints more precisely.
This is why the final energy is higher than the lowest energy reached during the first repetition, which was obtained with weak restraints.
The minimization is limited to 100 iterations per repetition: the aim is only to remove the worst clashes of atoms in the initial structure, not to find the exact minimum.
Because the minimization is fast and needs no input, there is no need to plot it for the other simulations in this tutorial.

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Which changes to the input structure can be made, for which the force field can still be applied? Try the following:
>
> - Remove an atom from the PDB file.
> - Remove an entire residue from the PDB file.
> - Use an engineered form of the Villin Headpiece, e.g. the Lys24Nle mutant (PDB 1WY3) or the Lys24Nle/Lys29Nle double mutant (PDB 2F4K).
>
> Finally, restore the input to the original one and rerun the code cell that sets up and minimizes the system, to have a good starting point for the next cell.
>
> With the methodology shown here, it is not possible to define custom mutations.

In [ ]:
simulation.reporters = []
simulation.reporters.append(DCDReporter("traj.dcd", 10))
simulation.reporters.append(
    StateDataReporter(stdout, 100, step=True, temperature=True, elapsedTime=True)
)
simulation.reporters.append(
    StateDataReporter(
        "scalars.csv",
        10,
        step=True,
        time=True,
        potentialEnergy=True,
        totalEnergy=True,
        temperature=True,
    )
)
simulation.step(30000)

# The last line is only needed for Windows users,
# to close the DCD file before it can be opened by nglview.
del simulation

You will need the output files of this simulation for the following notebook. Copy over the files `init.pdb`, `scalars.csv` and `traj.dcd` to the directory `../05_analysis`. This can be done with the following code, or with any file manager that comes with the operating system.

In [ ]:
shutil.copy("init.pdb", "../05_analysis")
shutil.copy("scalars.csv", "../05_analysis")
shutil.copy("traj.dcd", "../05_analysis")

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Make an estimate of the computational cost (wall time) on your current hardware to run a sufficiently long simulation to observe spontaneously a protein folding event for this fast folder. The required simulation time is approximately $4 \mathrm{\mu s}$. Would such a calculation be feasible?

In [ ]:
df = pandas.read_csv("scalars.csv")
plt.close("ener")
fig, ax = plt.subplots(num="ener")
df.plot(kind="line", x="Time (ps)", y="Potential Energy (kJ/mole)", ax=ax)

In [ ]:
traj = mdtraj.load("traj.dcd", top="init.pdb")
view = nglview.show_mdtraj(traj)
view